In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
from pathlib import Path

# Get the current working directory
CURRENT_DIR = Path.cwd()

print("Current working directory:")
print(CURRENT_DIR)

# If running from the notebooks folder
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = PROJECT_ROOT / "data" / "spam.csv"

print("\nProject root:")
print(PROJECT_ROOT)

print("\nDataset path:")
print(DATA_PATH)

Current working directory:
c:\Users\dan2s\OneDrive\Documents\ONE\Desktop\Fraud_SMS_Classifier\notebooks

Project root:
c:\Users\dan2s\OneDrive\Documents\ONE\Desktop\Fraud_SMS_Classifier

Dataset path:
c:\Users\dan2s\OneDrive\Documents\ONE\Desktop\Fraud_SMS_Classifier\data\spam.csv


In [5]:
if DATA_PATH.exists():
    print("Dataset found.")
else:
    raise FileNotFoundError(
        f"Dataset not found at:\n{DATA_PATH}"
    )

Dataset found.


In [26]:
import pandas as pd

encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

df = None

for encoding in encodings:
    try:
        df = pd.read_csv(DATA_PATH, encoding=encoding)
        print(f"Successfully loaded using '{encoding}' encoding.")
        break
    except UnicodeDecodeError:
        print(f"Failed with '{encoding}' encoding.")

if df is None:
    raise ValueError("Unable to read the dataset with the tested encodings.")

df.head()

Failed with 'utf-8' encoding.
Failed with 'utf-8-sig' encoding.
Successfully loaded using 'latin-1' encoding.


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN


In [9]:
df.tail()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
5567,spam,"This is the 2nd time we have tried 2 contact u. U have won the å£750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate.",NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other suggestions?",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free,NaN,NaN,NaN
5571,ham,Rofl. Its true to its name,NaN,NaN,NaN


In [10]:
rows, cols = df.shape

print(f"Rows    : {rows}")
print(f"Columns : {cols}")

Rows    : 5572
Columns : 5


In [11]:
print(df.columns.tolist())

['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 677.1 KB


In [13]:
df.describe(include="all")

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [14]:
missing = df.isnull().sum()

missing

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [15]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

Duplicate rows: 403


In [16]:
df = df.dropna(axis=1, how="all")

print(df.columns)

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')


In [17]:
df = df.rename(
    columns={
        "v1": "label",
        "v2": "message"
    }
)

df.head()

,label,message,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN


In [18]:
required_columns = ["label", "message"]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

print("Dataset structure validated.")

Dataset structure validated.


In [19]:
df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [20]:
print(df["label"].dtype)
print(df["message"].dtype)

str
str


In [21]:
df.sample(10, random_state=42)

,label,message,Unnamed: 2,Unnamed: 3,Unnamed: 4
3245,ham,"Funny fact Nobody teaches volcanoes 2 erupt, tsunamis 2 arise, hurricanes 2 sway aroundn no 1 teaches hw 2 choose a wife Natural disasters just happens",NaN,NaN,NaN
944,ham,"I sent my scores to sophas and i had to do secondary application for a few schools. I think if you are thinking of applying, do a research on cost also. Contact joke ogunrinde, her school is one me the less expensive ones",NaN,NaN,NaN
1044,spam,"We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p",NaN,NaN,NaN
2484,ham,Only if you promise your getting out as SOON as you can. And you'll text me in the morning to let me know you made it in ok.,NaN,NaN,NaN
812,spam,Congratulations ur awarded either å£500 of CD gift vouchers & Free entry 2 our å£100 weekly draw txt MUSIC to 87066 TnCs www.Ldew.com1win150ppmx3age16,NaN,NaN,NaN
2973,ham,"I'll text carlos and let you know, hang on",NaN,NaN,NaN
2991,ham,K.i did't see you.:)k:)where are you now?,NaN,NaN,NaN
2942,ham,No message..no responce..what happend?,NaN,NaN,NaN
230,ham,Get down in gandhipuram and walk to cross cut road. Right side &lt;#&gt; street road and turn at first right.,NaN,NaN,NaN
1181,ham,You flippin your shit yet?,NaN,NaN,NaN


In [22]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "spam_clean.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved cleaned dataset to:\n{OUTPUT_PATH}")

Saved cleaned dataset to:
c:\Users\dan2s\OneDrive\Documents\ONE\Desktop\Fraud_SMS_Classifier\data\spam_clean.csv


In [23]:
print("=" * 50)
print("DATA VALIDATION REPORT")
print("=" * 50)

print(f"Rows               : {df.shape[0]}")
print(f"Columns            : {df.shape[1]}")
print(f"Missing Values     : {df.isnull().sum().sum()}")
print(f"Duplicate Rows     : {df.duplicated().sum()}")
print(f"Spam Messages      : {(df['label'] == 'spam').sum()}")
print(f"Ham Messages       : {(df['label'] == 'ham').sum()}")

print("=" * 50)
print("Dataset is ready for Exploratory Data Analysis.")

DATA VALIDATION REPORT
Rows               : 5572
Columns            : 5
Missing Values     : 16648
Duplicate Rows     : 403
Spam Messages      : 747
Ham Messages       : 4825
Dataset is ready for Exploratory Data Analysis.
